# 나만의 스티어링 데이터셋 만들기

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/modulabs-personalab/psyctl/blob/main/examples/ko/03_generate_dataset.ipynb)

스티어링 벡터는 **대조 쌍(contrastive pairs)**으로 학습됩니다: 동일한 상황에 대해 "positive" 응답은 목표 성격을 나타내고, "neutral" 응답은 기본 응답입니다. PSYCTL은 [allenai/soda](https://huggingface.co/datasets/allenai/soda) 대화 데이터셋에서 이러한 쌍을 생성합니다.

**데이터 생성의 두 가지 경로:**
- **경로 A: OpenRouter API** -- GPU 불필요, 병렬 워커로 빠르게 생성, 50개 샘플 당 약 $0.01-0.05 비용
- **경로 B: 로컬 모델** -- 무료, GPU 필요, 느림

**이 노트북에서 다루는 내용:**
1. 대조 데이터셋의 구조를 이해합니다
2. OpenRouter API 또는 로컬 모델을 사용해 데이터셋을 생성합니다
3. 생성된 데이터를 검사합니다
4. (선택사항) HuggingFace Hub에 업로드합니다

**소요 시간:** 약 5분

In [ ]:
!pip install -q git+https://github.com/modulabs-personalab/psyctl.git bitsandbytes accelerate

In [ ]:
import os

try:
    from google.colab import userdata
    _hf = userdata.get("HF_TOKEN")
    os.environ["HF_TOKEN"] = _hf if isinstance(_hf, str) else str(_hf)
    print("Loaded HF_TOKEN from Colab Secrets")
except Exception:
    if not os.environ.get("HF_TOKEN"):
        raise RuntimeError(
            "HF_TOKEN not found. Please:\n"
            "1. Click the key icon (Secrets) in the left sidebar\n"
            "2. Add HF_TOKEN with your HuggingFace token\n"
            "3. Toggle 'Notebook access' ON for HF_TOKEN\n"
            "4. Re-run this cell"
        )

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PSYCTL_LOG_LEVEL"] = "WARNING"

# Optional: OpenRouter API key (for Path A)
try:
    from google.colab import userdata
    _or = userdata.get("OPENROUTER_API_KEY")
    os.environ["OPENROUTER_API_KEY"] = _or if isinstance(_or, str) else str(_or)
    print("Loaded OPENROUTER_API_KEY from Colab Secrets")
except Exception:
    pass

if not os.environ.get("OPENROUTER_API_KEY"):
    print("OPENROUTER_API_KEY not set. You can still use Path B (local model).")
    print("To set it: Add OPENROUTER_API_KEY to Colab Secrets, or enter it below.")

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PSYCTL_LOG_LEVEL"] = "WARNING"

## 데이터셋 구조

스티어링 데이터셋의 각 샘플은 다음 필드를 가진 JSONL 레코드입니다:

```json
{
  "situation": "Alice and Bob are at a coffee shop. Alice says: 'I can't believe...'",
  "char_name": "Bob",
  "positive": "Response exhibiting the target personality trait",
  "neutral": "Neutral baseline response"
}
```

**positive** 응답은 특정 성격(예: "극도로 우호적인")을 가진 캐릭터로 역할극하도록 프롬프트하여 생성되고, **neutral** 응답은 평범한 캐릭터 설명을 사용합니다. 이 두 활성화 간의 차이가 스티어링 벡터가 됩니다.

## 경로 A: OpenRouter API (GPU 불필요)

[OpenRouter](https://openrouter.ai/)를 통해 클라우드 모델을 사용합니다. 병렬 워커로 빠르게 생성할 수 있으며, API 키가 필요합니다.

In [ ]:
from pathlib import Path
from psyctl.core.dataset_builder import DatasetBuilder

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")

if OPENROUTER_API_KEY:
    OUTPUT_DIR = Path("./dataset_openrouter")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    builder = DatasetBuilder(
        use_openrouter=True,
        openrouter_api_key=OPENROUTER_API_KEY,
        openrouter_max_workers=4,
    )

    dataset_file = builder.build_caa_dataset(
        model="qwen/qwen3-next-80b-a3b-instruct",
        personality="Extroversion",
        output_dir=OUTPUT_DIR,
        limit_samples=10,
        dataset_name="allenai/soda",
        temperature=0.7,
        max_tokens=100,
    )
    print(f"Dataset generated: {dataset_file}")
else:
    print("Skipping Path A (no API key). Try Path B below.")

## 경로 B: 로컬 모델 (GPU 필요)

작은 로컬 모델을 사용합니다. 무료이지만 느리며, GPU 런타임이 필요합니다.

In [ ]:
OUTPUT_DIR_LOCAL = Path("./dataset_local")
OUTPUT_DIR_LOCAL.mkdir(parents=True, exist_ok=True)

builder_local = DatasetBuilder(use_openrouter=False)

dataset_file_local = builder_local.build_caa_dataset(
    model="google/gemma-3-270m-it",
    personality="Extroversion",
    output_dir=OUTPUT_DIR_LOCAL,
    limit_samples=10,
    dataset_name="allenai/soda",
    temperature=0.7,
    max_tokens=100,
)
print(f"Dataset generated: {dataset_file_local}")

## 데이터셋 검사

In [ ]:
import json
from pathlib import Path

# Use whichever dataset was generated above
dataset_path = None
for candidate in [Path("./dataset_openrouter"), Path("./dataset_local")]:
    if candidate.exists():
        jsonl_files = list(candidate.glob("*.jsonl"))
        if jsonl_files:
            dataset_path = jsonl_files[0]
            break

if dataset_path is None:
    print("No dataset found. Run Path A or Path B above first.")
else:
    samples = []
    with open(dataset_path, encoding="utf-8") as f:
        for line in f:
            samples.append(json.loads(line))

    print(f"File: {dataset_path}")
    print(f"Total samples: {len(samples)}")
    print(f"File size: {dataset_path.stat().st_size / 1024:.1f} KB")

    print("\n--- Sample 1 ---")
    s = samples[0]
    print(f"Character: {s['char_name']}")
    print(f"Situation: {s['situation'][:200]}...")
    print(f"Positive:  {s['positive']}")
    print(f"Neutral:   {s['neutral']}")

    if len(samples) > 1:
        print("\n--- Sample 2 ---")
        s = samples[1]
        print(f"Character: {s['char_name']}")
        print(f"Positive:  {s['positive']}")
        print(f"Neutral:   {s['neutral']}")

## (선택사항) HuggingFace Hub에 업로드

커뮤니티와 데이터셋을 공유합니다. **쓰기(write)** 권한이 있는 HuggingFace 토큰이 필요합니다.

In [ ]:
# Uncomment and modify to upload your dataset
#
# from psyctl.core.dataset_builder import DatasetBuilder
#
# builder = DatasetBuilder()
# repo_url = builder.upload_to_hub(
#     jsonl_file=dataset_path,
#     repo_id="your-username/extroversion-steering",  # Change this!
#     private=False,
#     license="mit",
# )
# print(f"Uploaded to: {repo_url}")